In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
from jaxcmr.summarize import (
    calculate_aic_weights,
    generate_t_p_matrices,
    summarize_parameters,
    winner_comparison_matrix,
    raw_winner_comparison_matrix,
    pairwise_aic_differences,
)

In [3]:
fit_tag = "50_set_likelihood_fixed_term_best_of_3"
fit_dir = "projects/TalmiEEG/results/fits/"
target_directory = "projects/TalmiEEG/"

data_names = [
    "TalmiEEG",
]

model_names = [
    "EEGMainEffects",
    "EEGMainEffectsPlusInteraction",
    # "EEGSimplerMainEffectsPlusInteraction",
    # "EEGEmotionOnly",
    # "EEGLPPOnly",
    "EEGLPPExponentOnly",
    "EEGLPPNonlinearInteraction",
    "EEGEmotionLPPExponentPlusInteraction",
]

model_titles = [
    "EEGMainEffects",
    "EEGMainEffectsPlusInteraction",
    # "EEGSimplerMainEffectsPlusInteraction",
    # "EEGEmotionOnly",
    # "EEGLPPOnly",
    "EEGLPPExponentOnly",
    "EEGLPPNonlinearInteraction",
    "EEGEmotionLPPExponentPlusInteraction",
]


query_parameters = [
    "encoding_drift_rate",
    "start_drift_rate",
    "recall_drift_rate",
    "shared_support",
    "item_support",
    "learning_rate",
    "primacy_scale",
    "primacy_decay",
    "stop_probability_scale",
    "stop_probability_growth",
    "choice_sensitivity",
    "emotion_scale",
    "lpp_main_scale",
    "lpp_main_threshold",
    "lpp_inter_scale",
    "lpp_inter_threshold",
    "lpp_main_exponent",
    "lpp_inter_exponent",
]


In [4]:
run_tag = "Model_Comparison"

if not model_titles:
    model_titles = model_names.copy()

for data_name in data_names:
    print(f"### {data_name}\n")
    results = []

    for model_name, model_title in zip(model_names, model_titles):
        fit_path = os.path.join(fit_dir, f"{data_name}_{model_name}_{fit_tag}.json")

        with open(fit_path) as f:
            results.append(json.load(f))
            if "subject" not in results[-1]["fits"]:
                results[-1]["fits"]["subject"] = results[-1]["subject"]
            if "allow_repeated_recalls" not in results[-1]["fixed"]:
                results[-1]["fixed"]["allow_repeated_recalls"] = False
                results[-1]["fits"]["allow_repeated_recalls"] = [False] * len(
                    results[-1]["fits"]["subject"]
                )
            results[-1]["name"] = model_title
            if "mfc_trace_sensitivity" in results[-1]["free"]:
                results[-1]["free"]["repetition_orthogonality"] = results[-1]["free"][
                    "mfc_trace_sensitivity"
                ]
                results[-1]["fits"]["repetition_orthogonality"] = results[-1]["fits"][
                    "mfc_trace_sensitivity"
                ]
                results[-1]["free"].pop("mfc_trace_sensitivity")
                results[-1]["fits"].pop("mfc_trace_sensitivity")

    summary = summarize_parameters(
        results, query_parameters, include_std=True, include_ci=True
    )

    table_path = os.path.join(
        target_directory, "tables", f"{data_name}_{fit_tag}_{run_tag}_parameters.md"
    )
    with open(table_path, "w") as f:
        f.write(summary)
    print("\nParameter Summary:")
    print(summary)

    df_t, df_p = generate_t_p_matrices(results)

    print("\nT-Test P-Value Matrix:")
    print(df_p.to_markdown())
    print()

    aic_weights = calculate_aic_weights(results)

    with open(
        os.path.join(
            target_directory,
            "tables",
            f"{data_name}_{fit_tag}_{run_tag}_aic_weights.md",
        ),
        "w",
    ) as f:
        f.write(aic_weights.to_markdown())

    print("\nAIC Weights:")
    print(aic_weights.to_markdown())
    print()

    df_comparison = winner_comparison_matrix(results)

    with open(
        os.path.join(
            target_directory,
            "tables",
            f"{data_name}_{fit_tag}_{run_tag}_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))

    print("\nWinner Ratios (AICw):")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    df_comparison = raw_winner_comparison_matrix(results)

    with open(
        os.path.join(
            target_directory,
            "tables",
            f"{data_name}_{fit_tag}_{run_tag}_raw_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))

    print("\nRaw Winner Ratios:")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    mean_delta, ci_halfwidth, _ = pairwise_aic_differences(results)

    delta_aic_table = mean_delta.copy().astype(object)
    for row_label in delta_aic_table.index:
        for col_label in delta_aic_table.columns:
            if row_label == col_label:
                delta_aic_table.loc[row_label, col_label] = ""
                continue
            mean_value = mean_delta.loc[row_label, col_label]
            ci = ci_halfwidth.loc[row_label, col_label]
            if mean_value != mean_value or ci != ci:
                delta_aic_table.loc[row_label, col_label] = ""
            else:
                lower = mean_value - ci
                upper = mean_value + ci
                delta_aic_table.loc[row_label, col_label] = f"{mean_value:.2f} [{lower:.2f}, {upper:.2f}]"

    print("\nPairwise ΔAIC (row − column; mean [CI]):")
    print(delta_aic_table.to_markdown())
    print()


### TalmiEEG


Parameter Summary:
| Parameter | Statistic | EEGMainEffects | EEGMainEffectsPlusInteraction | EEGLPPExponentOnly | EEGLPPNonlinearInteraction | EEGEmotionLPPExponentPlusInteraction |
|---|---|---|---|---|---|---|
| fitness | mean | 210.26 +/- 16.34 | 209.65 +/- 16.24 | 213.28 +/- 15.89 | 210.05 +/- 16.43 | 209.39 +/- 16.55 |
|  | std | 49.05 | 48.75 | 47.70 | 49.31 | 49.68 |
|  | min | 113.40 | 113.79 | 128.18 | 112.78 | 108.38 |
|  | max | 315.56 | 316.06 | 318.26 | 314.82 | 318.53 |
| encoding drift rate | mean | 0.61 +/- 0.11 | 0.65 +/- 0.11 | 0.60 +/- 0.10 | 0.56 +/- 0.10 | 0.55 +/- 0.11 |
|  | std | 0.33 | 0.33 | 0.29 | 0.30 | 0.33 |
|  | min | 0.00 | 0.00 | 0.06 | 0.01 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| start drift rate | mean | 0.37 +/- 0.13 | 0.36 +/- 0.13 | 0.46 +/- 0.13 | 0.40 +/- 0.14 | 0.44 +/- 0.14 |
|  | std | 0.39 | 0.38 | 0.38 | 0.41 | 0.42 |
|  | min | 0.00 | 0.00 | 0.00 | 0.00 | 0.00 |
|  | max | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
